某一間商店有 $N$ 件商品，其售價及成本分別為 $p_n$ 及 $c_n$， $n = 1, \cdots, N$。\
這些商品分別存放於對應的倉庫，倉庫的容量記為 $V_n$， $n = 1, \cdots, N$。\
每天商品的需求量服從 Poisson 分布，其期望值 $\lambda$ 僅依據當天是否為假日 (記為 $\lambda_{n,0}$ :平日；$\lambda_{n,1}$ :假日)。\
$\mathrm{Reward}(x) = -(x - 3)^2$，也就是與目標庫存量的方差，這推測會導致學習的成果與 linear regression 類似。

In [1]:
import numpy as np

In [2]:
# Parameter
N = 3
prices = [50 * n for n in range(1, N + 1)]
costs = [25 * n for n in range(1, N + 1)]
capacity = [5 + n for n in range(1, N + 1)]
lambda1 = [2 * n for n in range(1, N + 1)]
lambda0 = [n for n in range(1, N + 1)]
T = 100 # simulation time

In [3]:
expectRemainder = 3
def reward(x):
    return -(x - expectRemainder) ** 2

In [4]:
states = []
state = [0 for _ in range(N + 1)]
def generateStates(states, state, index):
    if index == N:
        state[index] = 0
        states.append(state.copy())
        state[index] = 1
        states.append(state.copy())
        return
    for i in range(capacity[index] + 1):
        state[index] = i
        generateStates(states, state, index + 1)
generateStates(states, state, 0)
statesLen = len(states)

In [5]:
stateActionPair = [[] for _ in range(statesLen)]
action = [0 for _ in range(N)]
def generateActions(stateActionPair, action, index, stateIndex):
    state = states[stateIndex]
    if index == N:
        stateActionPair[stateIndex].append(action.copy())
        return
    for i in range(capacity[index] - state[index] + 1):
        action[index] = i
        generateActions(stateActionPair, action, index + 1, stateIndex)
for stateIndex in range(statesLen):
    generateActions(stateActionPair, action, 0, stateIndex)

In [6]:
dim = 0
for i in range(statesLen):
    dim += len(stateActionPair[i])
print(dim)

90720


In [7]:
# Initialize
policy = [[0 for n in range(N)] for i in range(statesLen)]
avg = [[0 for j in range(len(stateActionPair[i]))] for i in range(statesLen)]
num = [[0 for j in range(len(stateActionPair[i]))] for i in range(statesLen)]
for t in range(T):
    for i in range(statesLen):
        state = states[i]
        maxValue = -1e6
        argmax = 0
        for j in range(len(stateActionPair[i])):
            action = stateActionPair[i][j]
            G = 0
            if state[-1] == 0:
                # weekday
                for n in range(N):
                    request = np.random.poisson(lambda0[n])
                    remainder = state[n] + action[n] - request
                    # G += prices[n] * min(request, state[n] + action[n])
                    # G -= costs[n] * action[n]
                    G += reward(remainder)
            elif state[-1] == 1:
                # holiday
                for n in range(N):
                    request = np.random.poisson(lambda1[n])
                    remainder = state[n] + action[n] - request
                    # G += prices[n] * min(request, state[n] + action[n])
                    # G -= costs[n] * action[n]
                    G += reward(remainder)
            num[i][j] += 1
            avg[i][j] = avg[i][j] + (G - avg[i][j]) / num[i][j]
            if avg[i][j] > maxValue:
                maxValue = avg[i][j]
                argmax = j
        for n in range(N):
            policy[i][n] = stateActionPair[i][argmax][n]

In [8]:
sample = np.random.randint(statesLen)
state = states[sample]
print(f"Sample state is {state}, policy is {policy[sample]}.")
if state[-1] == 0:
    expectRequest = lambda0
else:
    expectRequest = lambda1
print(f"Expected request is {expectRequest}.")
opitmal = [expectRemainder + exp for exp in expectRequest]
print(f"Optimal stock number is {opitmal}.")

Sample state is [1, 6, 1, 0], policy is [3, 0, 5].
Expected request is [1, 2, 3].
Optimal stock number is [4, 5, 6].
